In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== RUTAS =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1")
output_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
macro = pd.read_csv(base_dir / "Dataset Definitivo.csv")

# ===== ASEGURAR TIPOS =====
macro["price"] = pd.to_numeric(macro["price"], errors="coerce")
macro["rating"] = pd.to_numeric(macro["rating"], errors="coerce")
macro["review_count"] = pd.to_numeric(macro["review_count"], errors="coerce")
macro["subcategory"] = macro["subcategory"].astype("string").str.strip()

# ===== LIMPIEZA =====
df = macro.dropna(subset=["subcategory", "price"]).copy()

# ===== PERCENTILES PARA GAMA (65/90) =====
percentiles = (
    df.groupby("subcategory")["price"]
    .quantile([0.65, 0.90])
    .unstack()
    .rename(columns={0.65: "p65", 0.90: "p90"})
    .reset_index()
)

df = df.merge(percentiles, on="subcategory", how="left")

def asignar_gama(row):
    if pd.isna(row["price"]) or pd.isna(row["p65"]) or pd.isna(row["p90"]):
        return pd.NA
    if row["price"] >= row["p90"]:
        return "alta"
    elif row["price"] >= row["p65"]:
        return "media"
    else:
        return "baja"

df["price_tier"] = df.apply(asignar_gama, axis=1)

# ===== TOP 10% POPULARIDAD (MÉTODO CORREGIDO) =====
df = df.sort_values(['subcategory', 'review_count'], ascending=[True, False])
df['rank_popularity'] = df.groupby('subcategory').cumcount() + 1

subcategory_sizes = df.groupby('subcategory').size()
top10_thresholds = (subcategory_sizes * 0.10).apply(np.ceil).astype(int)
top10_thresholds = top10_thresholds.rename('top10_threshold').reset_index()

df = df.merge(top10_thresholds, on='subcategory', how='left')
df['top_10_popularity'] = df['rank_popularity'] <= df['top10_threshold']

q90_reviews = (
    df.groupby("subcategory")["review_count"]
    .quantile(0.90)
    .rename("q90_review_count")
    .reset_index()
)
df = df.merge(q90_reviews, on="subcategory", how="left")

# ===== TABLAS POR GAMA =====
h1_alta = df[df["price_tier"] == "alta"].copy()
h1_media = df[df["price_tier"] == "media"].copy()
h1_baja = df[df["price_tier"] == "baja"].copy()

# ===== RESUMEN =====
h1_summary = (
    df.groupby(["subcategory", "price_tier", "top_10_popularity"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_rating=("rating", "mean"),
        median_rating=("rating", "median"),
        avg_review_count=("review_count", "mean"),
        median_review_count=("review_count", "median"),
        avg_price=("price", "mean"),
        min_review_count=("review_count", "min"),
        max_review_count=("review_count", "max"),
    )
    .reset_index()
)

# ===== VALIDACIÓN =====
print("\n===== VALIDACIÓN DEL TOP 10% =====")
validation = df.groupby("subcategory").agg(
    total_productos=('product_name', 'count'),
    productos_top10=('top_10_popularity', 'sum'),
)
validation['pct_top10'] = (validation['productos_top10'] / validation['total_productos'] * 100).round(2)
print(validation)

print("\n===== DISTRIBUCIÓN POR GAMA =====")
print(df["price_tier"].value_counts())

print("\n===== VERIFICAR RATING =====")
print(f"Productos con rating: {df['rating'].notna().sum()} de {len(df)}")
print(f"Rating promedio: {df['rating'].mean():.2f}")
print(f"Rating mínimo: {df['rating'].min():.1f}, Máximo: {df['rating'].max():.1f}")

# ===== ANÁLISIS H1: COMPARACIÓN TOP 10% VS RESTO =====
print("\n" + "="*60)
print("ANÁLISIS HIPÓTESIS 1")
print("="*60)

# Global
top10_global = df[df['top_10_popularity'] == True]['rating'].mean()
resto_global = df[df['top_10_popularity'] == False]['rating'].mean()
diff_global = top10_global - resto_global

print(f"\n📊 COMPARACIÓN GLOBAL:")
print(f"  Rating promedio TOP 10%: {top10_global:.3f}")
print(f"  Rating promedio RESTO:   {resto_global:.3f}")
print(f"  Diferencia:              {diff_global:+.3f}")

# Por subcategoría
print(f"\n📊 COMPARACIÓN POR SUBCATEGORÍA:")
for subcat in df['subcategory'].unique():
    subcat_data = df[df['subcategory'] == subcat]
    top10_subcat = subcat_data[subcat_data['top_10_popularity'] == True]['rating'].mean()
    resto_subcat = subcat_data[subcat_data['top_10_popularity'] == False]['rating'].mean()
    diff_subcat = top10_subcat - resto_subcat
    
    print(f"\n  {subcat}:")
    print(f"    TOP 10%: {top10_subcat:.3f}")
    print(f"    RESTO:   {resto_subcat:.3f}")
    print(f"    Diferencia: {diff_subcat:+.3f}")

# Por gama de precio
print(f"\n📊 COMPARACIÓN POR GAMA DE PRECIO:")
for tier in ['baja', 'media', 'alta']:
    tier_data = df[df['price_tier'] == tier]
    if len(tier_data) > 0:
        top10_tier = tier_data[tier_data['top_10_popularity'] == True]['rating'].mean()
        resto_tier = tier_data[tier_data['top_10_popularity'] == False]['rating'].mean()
        diff_tier = top10_tier - resto_tier
        
        print(f"\n  Gama {tier.upper()}:")
        print(f"    TOP 10%: {top10_tier:.3f}")
        print(f"    RESTO:   {resto_tier:.3f}")
        print(f"    Diferencia: {diff_tier:+.3f}")

# ===== GUARDAR =====
df.to_csv(output_dir / "h1_dataset_completo.csv", index=False, encoding="utf-8-sig")
h1_alta.to_csv(output_dir / "h1_gama_alta.csv", index=False, encoding="utf-8-sig")
h1_media.to_csv(output_dir / "h1_gama_media.csv", index=False, encoding="utf-8-sig")
h1_baja.to_csv(output_dir / "h1_gama_baja.csv", index=False, encoding="utf-8-sig")
h1_summary.to_csv(output_dir / "h1_resumen.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"✅ Archivos guardados en: {output_dir}")
print(f"{'='*60}")


===== VALIDACIÓN DEL TOP 10% =====
             total_productos  productos_top10  pct_top10
subcategory                                             
Cascos                    40                4       10.0
Mandos                    40                4       10.0
Monitores                 40                4       10.0
PC                        31                4       12.9
Ratones                   40                4       10.0
Teclados                  40                4       10.0

===== DISTRIBUCIÓN POR GAMA =====
price_tier
baja     147
media     58
alta      26
Name: count, dtype: int64

===== VERIFICAR RATING =====
Productos con rating: 231 de 231
Rating promedio: 4.55
Rating mínimo: 3.0, Máximo: 5.0

ANÁLISIS HIPÓTESIS 1

📊 COMPARACIÓN GLOBAL:
  Rating promedio TOP 10%: 4.608
  Rating promedio RESTO:   4.542
  Diferencia:              +0.067

📊 COMPARACIÓN POR SUBCATEGORÍA:

  Cascos:
    TOP 10%: 4.475
    RESTO:   4.428
    Diferencia: +0.047

  Mandos:
    TOP 10%: 4.700


In [6]:
import pandas as pd
from pathlib import Path

# ===== CARGAR DATASET =====
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1")
df = pd.read_csv(output_dir / "h1_dataset_completo.csv")

print("\n" + "="*60)
print("ANÁLISIS H1: SUBCATEGORÍA × GAMA DE PRECIO")
print("="*60)

# Análisis detallado por subcategoría y gama
for subcat in sorted(df['subcategory'].unique()):
    print(f"\n{'='*60}")
    print(f"📦 {subcat.upper()}")
    print(f"{'='*60}")
    
    subcat_data = df[df['subcategory'] == subcat]
    
    for tier in ['baja', 'media', 'alta']:
        tier_data = subcat_data[subcat_data['price_tier'] == tier]
        
        if len(tier_data) == 0:
            continue
        
        # Filtrar top 10% y resto
        top10_tier = tier_data[tier_data['top_10_popularity'] == True]
        resto_tier = tier_data[tier_data['top_10_popularity'] == False]
        
        if len(top10_tier) > 0 and len(resto_tier) > 0:
            rating_top10 = top10_tier['rating'].mean()
            rating_resto = resto_tier['rating'].mean()
            diff = rating_top10 - rating_resto
            
            print(f"\n  Gama {tier.upper()}:")
            print(f"    Productos totales: {len(tier_data)}")
            print(f"    Top 10%: {len(top10_tier)} productos | Rating: {rating_top10:.3f}")
            print(f"    Resto:   {len(resto_tier)} productos | Rating: {rating_resto:.3f}")
            print(f"    Diferencia: {diff:+.3f}")
            
            # Reviews promedio
            reviews_top10 = top10_tier['review_count'].mean()
            reviews_resto = resto_tier['review_count'].mean()
            print(f"    Reviews promedio Top 10%: {reviews_top10:.0f}")
            print(f"    Reviews promedio Resto:   {reviews_resto:.0f}")
        
        elif len(top10_tier) > 0:
            print(f"\n  Gama {tier.upper()}:")
            print(f"    Solo productos Top 10%: {len(top10_tier)}")
            print(f"    Rating: {top10_tier['rating'].mean():.3f}")
        
        elif len(resto_tier) > 0:
            print(f"\n  Gama {tier.upper()}:")
            print(f"    Solo productos NO top 10%: {len(resto_tier)}")
            print(f"    Rating: {resto_tier['rating'].mean():.3f}")

# ===== CREAR TABLA RESUMEN =====
print("\n" + "="*60)
print("TABLA RESUMEN: RATING TOP 10% vs RESTO")
print("="*60)

resumen_list = []

for subcat in sorted(df['subcategory'].unique()):
    subcat_data = df[df['subcategory'] == subcat]
    
    for tier in ['baja', 'media', 'alta']:
        tier_data = subcat_data[subcat_data['price_tier'] == tier]
        
        if len(tier_data) == 0:
            continue
        
        top10_tier = tier_data[tier_data['top_10_popularity'] == True]
        resto_tier = tier_data[tier_data['top_10_popularity'] == False]
        
        if len(top10_tier) > 0 and len(resto_tier) > 0:
            resumen_list.append({
                'Subcategoría': subcat,
                'Gama': tier.upper(),
                'N_Total': len(tier_data),
                'N_Top10': len(top10_tier),
                'N_Resto': len(resto_tier),
                'Rating_Top10': round(top10_tier['rating'].mean(), 3),
                'Rating_Resto': round(resto_tier['rating'].mean(), 3),
                'Diferencia': round(top10_tier['rating'].mean() - resto_tier['rating'].mean(), 3),
                'Reviews_Top10': int(top10_tier['review_count'].mean()),
                'Reviews_Resto': int(resto_tier['review_count'].mean()),
            })

resumen_df = pd.DataFrame(resumen_list)
print("\n", resumen_df.to_string(index=False))

# Guardar tabla resumen
resumen_df.to_csv(output_dir / "h1_analisis_subcategoria_gama.csv", index=False, encoding="utf-8-sig")
print(f"\n✅ Tabla guardada en: {output_dir / 'h1_analisis_subcategoria_gama.csv'}")

# ===== IDENTIFICAR PATRONES =====
print("\n" + "="*60)
print("PATRONES IDENTIFICADOS")
print("="*60)

# Casos donde Top 10% tiene MEJOR rating
mejores = resumen_df[resumen_df['Diferencia'] > 0.05].sort_values('Diferencia', ascending=False)
print(f"\n✅ Top 10% con rating SIGNIFICATIVAMENTE MEJOR (>0.05):")
if len(mejores) > 0:
    for _, row in mejores.iterrows():
        print(f"  • {row['Subcategoría']} - Gama {row['Gama']}: +{row['Diferencia']:.3f} puntos")
else:
    print("  Ninguno")

# Casos donde Top 10% tiene PEOR rating
peores = resumen_df[resumen_df['Diferencia'] < -0.05].sort_values('Diferencia')
print(f"\n❌ Top 10% con rating PEOR (<-0.05):")
if len(peores) > 0:
    for _, row in peores.iterrows():
        print(f"  • {row['Subcategoría']} - Gama {row['Gama']}: {row['Diferencia']:.3f} puntos")
else:
    print("  Ninguno")

# Casos neutros
neutros = resumen_df[(resumen_df['Diferencia'] >= -0.05) & (resumen_df['Diferencia'] <= 0.05)]
print(f"\n⚪ Diferencia INSIGNIFICANTE (-0.05 a +0.05):")
if len(neutros) > 0:
    for _, row in neutros.iterrows():
        print(f"  • {row['Subcategoría']} - Gama {row['Gama']}: {row['Diferencia']:+.3f} puntos")
else:
    print("  Ninguno")


ANÁLISIS H1: SUBCATEGORÍA × GAMA DE PRECIO

📦 CASCOS

  Gama BAJA:
    Productos totales: 26
    Top 10%: 3 productos | Rating: 4.433
    Resto:   23 productos | Rating: 4.322
    Diferencia: +0.112
    Reviews promedio Top 10%: 4456
    Reviews promedio Resto:   880

  Gama MEDIA:
    Productos totales: 10
    Top 10%: 1 productos | Rating: 4.600
    Resto:   9 productos | Rating: 4.567
    Diferencia: +0.033
    Reviews promedio Top 10%: 3464
    Reviews promedio Resto:   1378

  Gama ALTA:
    Solo productos NO top 10%: 4
    Rating: 4.725

📦 MANDOS

  Gama BAJA:
    Solo productos NO top 10%: 24
    Rating: 4.521

  Gama MEDIA:
    Productos totales: 11
    Top 10%: 4 productos | Rating: 4.700
    Resto:   7 productos | Rating: 4.643
    Diferencia: +0.057
    Reviews promedio Top 10%: 2368
    Reviews promedio Resto:   159

  Gama ALTA:
    Solo productos NO top 10%: 5
    Rating: 4.680

📦 MONITORES

  Gama BAJA:
    Productos totales: 25
    Top 10%: 2 productos | Rating: 4.650
